# **XRAI: Practice**

Hi, everyone! As in the previous modules, we will reinforce the theory with practice. Next up is XRAI — a method that gives us not separate pixels, but pixels grouped into regions.

Moving from pixels to regions gives us details at a higher level of abstraction (you can draw a conclusion not about the influence of particular pixels, but about the influence of particular parts or objects of the image). In some cases this makes XRAI an excellent continuation of the gradient method it is based on (remember which one!).

In this practice you will
- Gain skills in working with the `saliency` library — the only (at the time of writing this course) framework-agnostic explanation library.
- Implement and analyse the results of the XRAI method
- Compare how informative the map produced by XRAI is against the map of a method that offers attributions without segments.

**As always — happy coding!**

<a href="https://ibb.co/N1Tq1bx"><img src="https://i.ibb.co/RhjKhd2/glen-carrie-4-Qr-Cip9gbx-Q-unsplash.jpg" alt="glen-carrie-4-Qr-Cip9gbx-Q-unsplash" border="0"></a>

In [ ]:
!pip3 uninstall numpy -y

In [ ]:
!pip3 install saliency torch torchvision captum numpy==1.26.4 -q # if needed, restart the notebook so that the correct version of numpy is installed

**While the required libraries are being installed, answer: which gradient method is XRAI based on?**

In [ ]:
# Required imports

import numpy as np
print(np.__version__)
import PIL.Image
from matplotlib import pylab as P
import matplotlib.pyplot as plt
import torch
import urllib
from torchvision import models, transforms

import saliency.core as saliency

from datetime import datetime

from captum.attr import IntegratedGradients
from captum.attr import visualization as viz

%matplotlib inline

from PIL import Image
from io import BytesIO
import requests

url = "https://raw.githubusercontent.com/pytorch/hub/master/imagenet_classes.txt"
urllib.request.urlretrieve(url, "imagenet_classes.txt")

with open("imagenet_classes.txt", "r") as f:
    categories = [s.strip() for s in f.readlines()]

As in the previous steps, let us first load an image. To make it more interesting, we will use a difficult image — it shows a cat sitting in a cupboard with dishes.

In [ ]:
cat_url = 'https://github.com/SadSabrina/explainable_AI_course/blob/a719daa485b85b94d4a60fff7f5e33ae39880461/HW_module12_concept%20based/cat_plates_cupboard.jpg?raw=true'

image_bytes = requests.get(cat_url).content
image = Image.open(BytesIO(image_bytes)) # load our cat

Let us prepare the preprocessing pipeline.

In [ ]:
# preprocessing of the image that we will feed to the model

to_tensor_transform = transforms.Compose([transforms.ToTensor(),
                                          transforms.Normalize(mean=[0.485, 0.456, 0.406],
                                                              std=[0.229, 0.224, 0.225])
])



# a simple resize of the original image, to keep it in the form we are used to
original_image = image.resize((256, 256))

In [ ]:
input_image = to_tensor_transform(original_image)

Let us visually check what the original and the transformed image look like.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 8))

ax[0].imshow(original_image)
ax[0].set_title('Original image')

ax[1].imshow(input_image.permute((1, 2, 0)))
ax[1].set_title('Image after transform')
plt.axis('off');

Now, as always, let us enable the return of gradients and add the batch dimension in order to feed the picture to the model. Do this on your own.

In [ ]:
# Your code here

**What is the sum of the dimensions of the input picture (`input image`) after adding the 4th dimension?**

Now let us load the model. We will take the already familiar `swin`.

In [ ]:
model = models.swin_t(weights='IMAGENET1K_V1')
model.eval();

### **Functions for XRAI**

The logic of the `saliency` library requires a strictly defined entity in order to run the XRAI algorithm — `call_model_function`. How the function should look is given in the official tutorial of the library and is laid out in the cell below. Let us figure out why it has to look exactly like this.

In [ ]:
def call_model_function(images, call_model_args=None, expected_keys=None):

   ### Preprocessing of the batch of images — start

    transformer = transforms.Normalize((0.485, 0.456, 0.406), (0.229, 0.224, 0.225))

    prep_images = np.transpose(images, (0,3,1,2))
    prep_images = torch.tensor(prep_images, dtype=torch.float32)
    prep_images = transformer.forward(prep_images)
    prep_images.requires_grad_(True)

    ### Preprocessing of the batch of images — end

    target_class_idx = call_model_args[class_idx_str]
    output = model(prep_images) # get the output of the model
    m = torch.nn.Softmax(dim=1) # pass all the values through softmax
    output = m(output)

    if saliency.base.INPUT_OUTPUT_GRADIENTS in expected_keys:
        outputs = output[:,target_class_idx]
        grads = torch.autograd.grad(outputs, prep_images, grad_outputs=torch.ones_like(outputs))
        grads = torch.movedim(grads[0], 1, 3)
        gradients = grads.detach().numpy()
        return {saliency.base.INPUT_OUTPUT_GRADIENTS: gradients}

    else:
        one_hot = torch.zeros_like(output)
        one_hot[:,target_class_idx] = 1
        model.zero_grad()
        output.backward(gradient=one_hot, retain_graph=True)
        return conv_layer_outputs

- `images`:
   - must be given strictly as a numpy array, because when the masks are extracted the dimensionality of the image is checked, which requires it to have the `shape` attribute;
   - must contain the original image;

- The presence of a logical condition (if-else):
  - allows this function to be used universally for any attribution method in the `saliency` library. Specifically, when using the XRAI method you fall under the if branch.

**Question: what does `call_model_function` return for the XRAI method?**

Now let us get the prediction of the model and verify the top-5 predictions. Do this on your own.

In [ ]:
predictions = # Your code here

softmax = torch.nn.Softmax(dim=1) # let us also pass all the values through softmax
predictions_probas = softmax(predictions)

# Extract the top-5 and the top-1

top5 = # Your code here

predictions_probas = # Your code here
prediction_class = np.argmax(predictions_probas[0])

print("Prediction class: " + str(prediction_class))

In [ ]:
top5_indexes = top5.indices.detach().numpy()[0]
top5_values = # Your code here

for idx, value in zip(top5_indexes, top5_values):
  print(f'Categorie: "{categories[idx]}" with idx {idx}, value={round(value, ndigits=2)}')

**What is the probability of the class number 4, rounded to one decimal place?**

You can notice that the attention of the model is "sharpened" on the top-3. Now it is time to look at which details the model pays attention to when making such a choice, with the help of XRAI!

In [ ]:
# The attention of the model when predicting class 495

class_idx_str = 'class_idx_str'
class_interested = 495 # Choose the predicted class as the class to explain and put it into call_model_args


call_model_args = {class_idx_str: prediction_class}

start_time = datetime.now()
# Construct the saliency object. This alone doesn't do anthing.
xrai_object = saliency.XRAI()


# Compute XRAI attributions with default parameters
xrai_attributions = xrai_object.GetMask(np.array(original_image), call_model_function, call_model_args, batch_size=1)

print('Computation time:', datetime.now() - start_time)

Let us visualise the result of the feature attribution obtained with XRAI.

In [ ]:
# Show most salient 10% of the image
mask = xrai_attributions >= np.percentile(xrai_attributions, 90)
im_mask = np.array(original_image)
im_mask[~mask] = 0

fig, ax = plt.subplots(1, 3, figsize=(18, 8))

ax[0].imshow(original_image)
ax[0].set_title('Original image')

ax[1].imshow(xrai_attributions, cmap='inferno')
ax[1].set_title('Image after segmentation')

ax[2].imshow(im_mask)
ax[2].set_title('Top-10%')
plt.axis('off');

The model "pays attention" to the dishes and predicts quite a natural class — a cupboard for dishes (china cabinet). Let us look at how the "attention" of the model looks through the prism of Integrated Gradients.

In [ ]:
integrated_gradients = IntegratedGradients(model)
model.zero_grad()
captum_ig_attributions = integrated_gradients.attribute(inputs=input_image,
                                                        baselines=input_image * 0,
                                                        target=495,
                                                        n_steps=50,
                                                        method='riemann_right')


In [ ]:
plt.imshow(captum_ig_attributions.squeeze(0).permute((1, 2, 0)).detach().numpy()*10)
plt.title('Integrated gradients attributuion');

As we can see, the result is similar, however the level of detail given by Integrated Gradients also lets us see that the model "sharpens its attention" on the borders of the cupboard (possibly on the presence of the wall), which the segmentation does not show.

**Build the maps for the target class (285, Egyptian cat) with both methods and analyse the result. Which attribution method seems to you to fit better here?**

**Formulate your conclusions and write them as a short text (an essay).**

In [ ]:
# The attention of the model when predicting class 285

class_idx_str = 'class_idx_str'
class_interested = 285 # Choose the predicted class as the class to explain and put it into call_model_args


call_model_args = {class_idx_str: prediction_class}

start_time = datetime.now()
# Construct the saliency object. This alone doesn't do anthing.
xrai_object = saliency.XRAI()


# Compute XRAI attributions with default parameters
xrai_attributions = xrai_object.GetMask(np.array(original_image), call_model_function, call_model_args, batch_size=1)

print('Computation time:', datetime.now() - start_time)

In [ ]:
# The top 10 most important regions for the model
mask = xrai_attributions >= np.percentile(xrai_attributions, 85)
im_mask = np.array(original_image)
im_mask[~mask] = 0

fig, ax = plt.subplots(1, 3, figsize=(18, 8))

ax[0].imshow(original_image)
ax[0].set_title('Original image')

ax[1].imshow(xrai_attributions, cmap='inferno')
ax[1].set_title('Image after segmentation')

ax[2].imshow(im_mask)
ax[2].set_title('Top-15%')
plt.axis('off');

In [ ]:
integrated_gradients = IntegratedGradients(model)
model.zero_grad()
captum_ig_attributions = integrated_gradients.attribute(inputs=input_image,
                                                        baselines=input_image * 0,
                                                        target=285,
                                                        n_steps=50,
                                                        method='riemann_right')


In [ ]:
plt.imshow(captum_ig_attributions.squeeze(0).permute((1, 2, 0)).detach().numpy()*10)
plt.title('Integrated gradients attributuion');

As the last point we would like to draw your attention to, let us show that there is a way to build XRAI maps faster.

1. The first one is to implement a custom segmentation or to use already annotated images
2. The second one is to use the fast variant of the algorithm, but here it is important to keep in mind that it gives an approximate result, which may differ from XRAI without the speed-up, and it does not let you win a lot of time on all images.

In [ ]:
# The attention of the model when predicting class 285

class_idx_str = 'class_idx_str'
class_interested = 285 # Choose the predicted class as the class to explain and put it into call_model_args


call_model_args = {class_idx_str: prediction_class}

start_time = datetime.now()

# Create XRAIParameters and set the algorithm to fast mode which will produce an approximate result.
# Construct the saliency object. This alone doesn't do anthing.
xrai_object = saliency.XRAI()

xrai_params = saliency.XRAIParameters()
xrai_params.algorithm = 'fast'

# Compute XRAI attributions with fast algorithm
xrai_attributions_fast = xrai_object.GetMask(np.array(original_image), call_model_function, call_model_args, extra_parameters=xrai_params, batch_size=1)

print('Computation time:', datetime.now() - start_time)

In [ ]:
# Let us highlight the top-15 most important pixels
mask = xrai_attributions >= np.percentile(xrai_attributions, 85)
im_mask = np.array(original_image)
im_mask[~mask] = 0

fig, ax = plt.subplots(1, 3, figsize=(18, 8))

ax[0].imshow(original_image)
ax[0].set_title('Original image')

ax[1].imshow(xrai_attributions, cmap='inferno')
ax[1].set_title('Image after segmentation')

ax[2].imshow(im_mask)
ax[2].set_title('Top-15%')
plt.axis('off');

**Thank you for your work! We are looking forward to your essays! :)**